Q1: Modeling and Analyzing Neural Spike Patterns
A. Constructing the Models and Assumptions

I simulated 5 seconds of spike times for three types of neurons. I chose parameters so that all three neurons fire at about the same average speed (~10 spikes per second), so I can compare their patterns fairly.

1. Regular neuron: This model is deterministic (no randomness). It fires at a fixed interval of 0.1 seconds.

2. Irregular neuron: This model is stochastic (random). Each waiting time between spikes is picked randomly from an exponential distribution (rng.exponential) with an average of 0.1 seconds. I used seed=42 so I get the same results every time I run the notebook.

3. Bursting neuron: This model fires 4 quick spikes in a group (about 0.02 seconds apart) and then waits a longer quiet time (about 0.34 seconds) before the next group. I added small random noise to both waiting times so the bursts are not identical.

To make sure all spike times are valid and in time order, I always add positive waiting times and stop at 5.0 seconds. I also used max() so the random noise never makes an interval zero or negative.

In [ ]:
import numpy as np
# start - 0.1 , stop - 5 , step - 0.1
regular_spikes = np.arange(0.1, 5.01, 0.1)
print("Count:", len(regular_spikes))
print("First 5 spikes:", regular_spikes[:5])    



Count: 50
First 5 spikes: [0.1 0.2 0.3 0.4 0.5]


In [4]:
# Create a random generator with a fixed seed for reproducible results
rng = np.random.default_rng(42)

spikes = []          # Empty list to store spike times
current_time = 0.0   # Start clock at 0 seconds

while True:
    # Draw a random positive waiting time (mean = 0.1 s)
    isi = rng.exponential(scale=0.1)
    current_time += isi  # Move clock forward

    # Stop if we pass the 5-second recording window
    if current_time > 5.0:
        break

    spikes.append(current_time)  # Save valid spike time

# Convert list to NumPy array for easy calculations later
irregular_spikes = np.array(spikes)

print("Count:", len(irregular_spikes))
print("First 5 spikes:", np.round(irregular_spikes[:5], 3))

Count: 60
First 5 spikes: [0.24  0.474 0.713 0.74  0.749]


In [ ]:
# I chose 4 spikes per burst so each group of spikes is easy to see
spikes_per_burst = 4
intra_burst_mean = 0.02  # Short interval within a burst (20 ms)
inter_burst_mean = 0.34  # Long quiet interval between bursts (340 ms)

spikes_burst = []
current_time = 0.05  # Start after t = 0 for consistency with the other neurons and clearer plotting

while current_time <= 5.0:
    # 1. Fire a rapid burst of 4 spikes
    for i in range(spikes_per_burst):
        if current_time > 5.0:
            break
        spikes_burst.append(current_time)
        # Add a short gap only between spikes in the burst (not after the last spike)
        if i < spikes_per_burst - 1:
        # Keep the interval positive (at least 2 ms) even if random noise is negative. 0.002 is for safety
            intra_isi = max(0.002, intra_burst_mean + rng.normal(0, 0.003))
            current_time += intra_isi

    # 2. Add a long quiet interval before the next burst starts
        # Keep the quiet gap between bursts positive (at least 50 ms) even if random noise is negative
    inter_isi = max(0.05, inter_burst_mean + rng.normal(0, 0.015))
    current_time += inter_isi

bursting_spikes = np.array(spikes_burst)

print("Count:", len(bursting_spikes))
print("First 8 spikes:", np.round(bursting_spikes[:8], 3))

Count: 52
First 8 spikes: [0.05  0.073 0.095 0.119 0.456 0.474 0.494 0.514]
